# Hugging Face Basics + Fine-Tuning (bare minimum)

**Flow of this notebook:**

1. **Hugging Face Basics** — load a dataset, print a sample, load a model via a pipeline, run it.
2. **Fine-Tuning (bare minimum)** — BERT classification fine-tuning, and generation-model fine-tuning with **LoRA** and **QLoRA**.

> Adapted from the original Colab notebooks (Hugging Face tutorial, and *Hands-On Large Language Models* chapters 11/12).

This notebook downloads and runs local models — Part 2 (fine-tuning) needs a CUDA GPU, particularly the QLoRA section (4-bit quantization via `bitsandbytes` doesn't run on CPU).


## 0. Setup

**Note on the install command below:** package version specifiers like `transformers>=4.41.2` need to be wrapped in quotes when run through `!pip install` in a notebook. Without quotes, the shell reads `>` as an output-redirect operator, silently drops the version constraint, and dumps stray files in your folder instead of actually enforcing it. Quoting (`"package>=1.0"`) avoids that.

`datasets` is capped below 3.0 deliberately — the 3.x/5.x line has had breaking changes that crash on import with some dependency combinations; the 2.x line is stable for everything used here.


In [ ]:
!pip install -q torch "transformers>=4.41.2" "datasets>=2.18.0,<3.0.0" "huggingface_hub>=0.23.0" "accelerate>=0.31.0" sentencepiece evaluate
!pip install -q "peft>=0.11.1" "bitsandbytes>=0.43.1" "trl>=0.9.4"


In [ ]:
import torch

# Check GPU availability — Part 2's QLoRA section needs a CUDA GPU to actually run.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


## Part 1: Hugging Face Basics

Load a dataset, print a sample, load a pretrained model through a `pipeline`, and run it.

*(Adapted from `Tutorial_1_Hugging_Face_Final_Version.ipynb`)*


In [ ]:
from datasets import load_dataset

# Load a dataset and print a sample
dataset = load_dataset("stanfordnlp/imdb", "plain_text", split="train")
print(dataset[0])


In [ ]:
from transformers import pipeline

# Load a pretrained model via a pipeline and run it
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
result = sentiment_pipeline(dataset[0]["text"][:512])
print(result)


## Part 2: Fine-Tuning (bare minimum)

Two bare-minimum fine-tuning examples: classification fine-tuning with BERT, and generation-model fine-tuning with **LoRA** / **QLoRA**. These are trimmed for demonstration (small samples, few epochs) — not full training runs.

*(Adapted from `Chapter_11_-_Fine-Tuning_BERT.ipynb` and `Chapter_12_-_Fine-tuning_Generation_Models.ipynb`.)*

### 2.1 BERT Fine-Tuning (classification, bare minimum)


In [ ]:
from datasets import load_dataset

# Prepare data and splits
tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Model and Tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)


In [ ]:
from transformers import DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data (small subset for a quick demo)
tokenized_train = train_data.select(range(200)).map(preprocess_function, batched=True)
tokenized_test = test_data.select(range(100)).map(preprocess_function, batched=True)


In [ ]:
from transformers import TrainingArguments, Trainer

# Training arguments for a quick demo run
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="no",
   report_to="none"
)

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
)

trainer.train()
trainer.evaluate()


### 2.2 Generation Model Fine-Tuning with LoRA / QLoRA (bare minimum)

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    """Format the prompt using the <|user|> template TinyLlama uses"""
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

# Small subset for a quick demo (original tutorial uses 3,000 examples)
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")
      .shuffle(seed=42)
      .select(range(200))
)
dataset = dataset.map(format_prompt)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-bit quantization configuration - the "Q" in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,             # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",     # Quantization type
    bnb_4bit_compute_dtype="float16",  # Compute dtype
    bnb_4bit_use_double_quant=True,    # Apply nested quantization
)

# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",

    # Leave this out for regular LoRA (no quantization) instead of QLoRA
    quantization_config=bnb_config,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# LoRA configuration
peft_config = LoraConfig(
    lora_alpha=32,      # LoRA scaling
    lora_dropout=0.1,   # Dropout for LoRA layers
    r=64,               # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=      # Layers to target
     ['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)


In [ ]:
from transformers import TrainingArguments

output_dir = "./results"

# Training arguments for a quick demo run
training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True
)


In [ ]:
from trl import SFTTrainer

# Supervised fine-tuning with the LoRA/QLoRA config
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    max_seq_length=512,

    # Leave this out for regular SFT (no LoRA)
    peft_config=peft_config,
)

# Train model
trainer.train()

# Save QLoRA weights
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")


In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA weights into the base model
merged_model = model.merge_and_unload()


In [ ]:
from transformers import pipeline

# Use our predefined prompt template
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])
